# How the TradeMatrix model is tested (BTC, 1h)

*Created by Argy*. This notebook rebuilds `reports/backtest_BTC_1h_*_expanding_strongreg.md` step by step,
so every number in the report can be traced. Terms are explained in `docs/07-LEARNING-GUIDE.md`.

Run it from `backend/`: `uv run --with jupyter jupyter lab notebooks/`. It reads the candles from the
database in the root `.env`.

In [ ]:
import sys

sys.path.insert(0, "..")  # so `import app` works from the notebooks/ folder

import numpy as np
import pandas as pd

from app.features.build import feature_columns
from app.ml import metrics
from app.ml.backtest import load_jobs, run_walk_forward, summarize
from app.ml.config import EVAL_FROM, chosen
from app.ml.train import fit_calibrated
from app.ml.walkforward import walk_forward

## 1. Data: one row per closed candle

Each row holds the features known when candle *t* closed, and the **label**: did candle *t+1* close
higher (1) or not (0)? `prev_up` is the direction of candle *t* itself, used by the naive baseline.

In [ ]:
((name, data),) = await load_jobs(["BTC"], "1h")
print(f"{len(data):,} rows from {data.index[0]:%Y-%m-%d} to {data.index[-1]:%Y-%m-%d}")
print(f"share of candles that closed higher: {data['label'].mean():.3f}")
data[["rsi14", "dist_ema50", "macd_hist_pct", "vol_ratio_20", "label"]].tail()

## 2. The baselines the model must beat

- **naive**: say the next candle repeats the last one's direction.
- **always up**: always say Up.

If a model cannot beat these simple rules, its signal is worthless, however clever it looks.

In [ ]:
naive = (data["prev_up"] == data["label"]).mean()
always_up = data["label"].mean()
print(f"naive accuracy: {naive:.3f}   always-up accuracy: {always_up:.3f}")
# On 1h crypto the naive rule is right less than half the time: short moves tend to reverse a little.

## 3. Walk-forward folds

Train on the past, test on the next month, roll forward. Validation is split in two halves:
the first decides when to stop adding trees, the second calibrates the probabilities.

In [ ]:
config = chosen("1h")
folds = walk_forward(data.index, **config.windows, expanding=config.expanding)
print(f"settings: {config.name}, {len(folds)} folds")
first = folds[0]
for part in ("train", "early_stop", "calibrate", "test"):
    rows = getattr(first, part)
    print(
        f"{part:10} {data.index[rows[0]]:%Y-%m-%d} .. {data.index[rows[-1]]:%Y-%m-%d} ({len(rows):,} rows)"
    )

## 4. One fold by hand

Train, predict the test month, then compare with the baselines on exactly the same rows.

In [ ]:
fold = folds[-1]
features = feature_columns(data)
model = fit_calibrated(
    data.iloc[fold.train],
    data.iloc[fold.early_stop],
    data.iloc[fold.calibrate],
    features,
    config.params,
)
test = data.iloc[fold.test]
p = model.predict(test)
y = test["label"].to_numpy()
result = metrics.classification(p, y)
print(
    f"calls made on {result['coverage']:.1%} of candles, right {result['accuracy']:.1%} of the time"
)
print(f"naive on the same month: {(test['prev_up'].to_numpy() == y).mean():.1%}")
print(
    f"Brier: model {result['brier']:.4f} vs base rate {metrics.brier(np.full(len(y), data.iloc[fold.train]['label'].mean()), y):.4f}"
)
pd.DataFrame(metrics.reliability_table(p, y))

## 5. The full evaluation (the numbers in the report)

Only folds whose test month starts on or after `EVAL_FROM` count. The settings were chosen
earlier, on 2021-2024 only (`reports/tuning_*.md`), so these months played no part in the choice.

In [ ]:
preds, fold_rows = run_walk_forward(data, "1h", config, EVAL_FROM)
summary = summarize(preds, fold_rows, "1h")
m, b = summary["model"], summary["baselines"]
print(
    f"accuracy on calls {m['accuracy']:.1%} (coverage {m['coverage']:.1%}) vs naive {b['naive']:.1%}"
)
print(f"Brier {m['brier']:.4f} vs base rate {b['brier_base_rate']:.4f}")
print(
    "accuracy edge vs naive, mean and 95% interval:", np.round(summary["accuracy_edge_vs_naive"], 4)
)
print(
    "Brier gain vs base rate, mean and 95% interval:",
    np.round(summary["brier_edge_vs_base_rate"], 5),
)
print("beats both baselines:", summary["beats_baselines"])
pd.DataFrame(summary["strategy"]).T

## What to take away

- The edge is real (both intervals stay above zero) but **small**.
- Trading every hourly Up call still loses money after fees: the edge is smaller than trading costs.
  TradeMatrix shows a probability to help a decision; it is not a trading system.
- If a result ever looks much better than this, assume a bug (data leakage) first.